# P16A — Pembangkit citra retak sintetis
Notebook ini membangkitkan 2.400 citra abu-abu berukuran 32 × 32 piksel dari 60 panel beton virtual (40 citra per panel). Separuh citra memuat retak berupa lintasan acak yang lebih gelap dari latar. Dua pengecoh sengaja ditambahkan pada sebagian citra: garis sambungan lurus dan bercak gelap kecil, sehingga "ada garis gelap" saja belum cukup untuk menyatakan retak.

Setiap panel punya kecerahan, pola noda, dan tekstur sendiri. Karena itu panel adalah unit kelompok untuk pembagian data (lihat P16B). Citra sintetis ini untuk latihan membaca alur deep learning; bukan citra lapangan dan tidak dapat dipakai untuk menilai kinerja deteksi retak sebenarnya. *Seed* dicatat di berkas keluaran.


In [ ]:
# Folder kerja: jalankan dari folder mana pun. Sel ini pindah ke akar repositori agar jalur data relatif (praktikum/...) benar.
import os
from pathlib import Path
_akar = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "praktikum").is_dir() and (p / "notebooks").is_dir()), None)
if _akar is not None:
    os.chdir(_akar)
else:
    print("Akar repositori tidak ditemukan; memakai folder kerja saat ini (normal di Colab).")

In [ ]:
from pathlib import Path
import numpy as np
from scipy.ndimage import gaussian_filter

SEED = 16016
N_PANEL, PER_PANEL, S = 60, 40, 32          # 60 panel x 40 citra, 32 x 32 piksel
rng = np.random.default_rng(SEED)
out = Path("praktikum/bab16"); out.mkdir(parents=True, exist_ok=True)

def gambar_retak(img, r):
    """Retak = lintasan acak berkelanjutan, 1-2 piksel, lebih gelap dari latar.
    Lintasan diulang bila kurang dari 18 piksel yang tampak di dalam citra."""
    while True:
        x, y = r.uniform(0, S), r.uniform(0, S)
        sudut = r.uniform(0, 2*np.pi); panjang = r.integers(22, 45)
        lebar = r.integers(1, 3); kontras = r.uniform(35, 70)
        mask = np.zeros((S, S))
        for _ in range(panjang):
            sudut += r.normal(0, 0.35)
            x += np.cos(sudut); y += np.sin(sudut)
            i, j = int(round(y)), int(round(x))
            if 0 <= i < S and 0 <= j < S:
                mask[max(i-lebar//2, 0):i+lebar//2+1, max(j-lebar//2, 0):j+lebar//2+1] = 1
        if mask.sum() >= 18: break
    return img - kontras * gaussian_filter(mask, 0.6)

def gambar_sambungan(img, r):
    """Garis sambungan lurus penuh: pengecoh yang bukan retak."""
    k = r.integers(4, S-4); kontras = r.uniform(12, 25)
    if r.random() < 0.5: img[k, :] -= kontras
    else: img[:, k] -= kontras
    return img

def gambar_bercak(img, r):
    """Bercak gelap kecil (lubang atau agregat) sebagai pengecoh kedua."""
    for _ in range(r.integers(1, 4)):
        ci, cj = r.integers(3, S-3), r.integers(3, S-3); rad = r.uniform(1.2, 2.6)
        ii, jj = np.ogrid[:S, :S]
        img = img - r.uniform(25, 55) * np.exp(-((ii-ci)**2 + (jj-cj)**2) / (2*rad**2))
    return img

X, y, panel = [], [], []
for p in range(N_PANEL):
    kecerahan = rng.uniform(105, 150); amp_halus = rng.uniform(7, 16)
    noda = gaussian_filter(rng.normal(0, 1, (S, S)), 6) * rng.uniform(20, 45)   # pola panel (noda, cahaya)
    for _ in range(PER_PANEL):
        img = kecerahan + noda + amp_halus * gaussian_filter(rng.normal(0, 1, (S, S)), 1.0) * 2.2
        ada = int(rng.random() < 0.5)
        if ada: img = gambar_retak(img, rng)
        if rng.random() < 0.30: img = gambar_sambungan(img, rng)
        if rng.random() < 0.30: img = gambar_bercak(img, rng)
        img = img + rng.normal(0, 3, (S, S))
        X.append(np.clip(img, 0, 255).astype(np.uint8)); y.append(ada); panel.append(p)

X, y, panel = np.stack(X), np.array(y), np.array(panel)
np.savez_compressed(out / "citra_retak_sintetis.npz", X=X, y=y, panel=panel, seed=SEED)
print("Citra:", X.shape, "| dtype:", X.dtype, "| panel:", len(np.unique(panel)))
print("Proporsi retak:", round(float(y.mean()), 3), "| rentang piksel:", int(X.min()), "-", int(X.max()))
print("Disimpan:", out / "citra_retak_sintetis.npz")


Keluaran: `praktikum/bab16/citra_retak_sintetis.npz` dengan larik `X` (citra, uint8), `y` (1 = retak), `panel` (nomor panel), dan `seed`. Menjalankan ulang dengan `SEED` yang sama menghasilkan berkas yang identik.
